# QM640 Capstone — Medicare Provider Data Quality & Exploratory Data Analysis

**Project:** Predicting Medicare Payment Costs and Identifying High-Cost Provider Patterns Using Machine Learning  
**Course:** QM640 — Data Analytics Capstone | **Mentor:** Sharath Srivatsa | **Term:** Fall 2026

This notebook adapts the supplied *Capstone_ExploratoryDataAnalysis.ipynb* workflow (context, data description, sanity checks, missingness, distributions, outlier review, and exports) to the 2024 CMS Medicare Physician & Other Practitioners — by Provider dataset.

**Important:** This notebook is exploratory. It does not train models or assert that high payments are improper. Do not replace CMS-suppressed/missing values with zero, and do not automatically cap or delete high payment values.

## 1. Context and project boundaries

The unit of analysis is a provider/NPI-level aggregate record. The target planned for later modeling is `Tot_Mdcr_Pymt_Amt`. The data represent the reporting scope described by CMS; results should not be generalized to individual claims/patients, Medicare Advantage, Medicaid, commercial insurance, or total provider practice revenue.

The EDA is intended to answer:
- What is the actual file structure, row count, schema, and key uniqueness?
- Where are missing, suppressed, or top-coded values concentrated?
- Are there duplicate records, impossible numeric values, or internal consistency issues?
- How skewed is provider payment, and what candidate percentile thresholds could define “high cost”?
- Which candidate predictors are strongly associated with the target and may introduce target leakage?

CMS sources:
- Dataset: https://data.cms.gov/provider-summary-by-type-of-service/medicare-physician-other-practitioners/medicare-physician-other-practitioners-by-provider
- API docs: https://data.cms.gov/provider-summary-by-type-of-service/medicare-physician-other-practitioners/medicare-physician-other-practitioners-by-provider/api-docs
- Data dictionary: https://data.cms.gov/resources/medicare-physician-other-practitioners-by-provider-data-dictionary

## 2. Import libraries and set options

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
warnings.filterwarnings("ignore", category=FutureWarning)

OUTPUT_DIR = Path("reports/eda")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print("Output directory:", OUTPUT_DIR.resolve())

## 3. Load the CMS file

Set `DATA_PATH` to the downloaded CSV. In Google Colab, upload the file or mount Drive and update this path. Avoid hard-coded personal Drive paths in the public repository.

In [ ]:
# Update this path for your environment.
DATA_PATH = Path("data/raw/cms_provider_2024.csv")

# If running in Colab and the file was uploaded to the session, you may use:
# DATA_PATH = Path("/content/cms_provider_2024.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"CSV not found at {DATA_PATH.resolve()}. Download the official CMS file, "
        "place it at data/raw/cms_provider_2024.csv, or update DATA_PATH."
    )

data = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Rows: {data.shape[0]:,} | Columns: {data.shape[1]:,}")
display(data.head())
display(data.tail())

## 4. Dataset structure and schema

Check row/column count, column names, inferred types, memory use, and examples. Do not assume the schema matches a previous download; record the actual schema of the file used for each run.

In [ ]:
schema = pd.DataFrame({
    "column": data.columns,
    "dtype": [str(data[c].dtype) for c in data.columns],
    "non_null_count": [int(data[c].notna().sum()) for c in data.columns],
    "missing_count": [int(data[c].isna().sum()) for c in data.columns],
    "missing_pct": [float(data[c].isna().mean() * 100) for c in data.columns],
    "n_unique_non_null": [int(data[c].nunique(dropna=True)) for c in data.columns],
})
display(schema.sort_values(["missing_pct", "column"], ascending=[False, True]))
schema.to_csv(OUTPUT_DIR / "column_profile.csv", index=False)

data.info(memory_usage="deep")

## 5. Sanity checks: keys, duplicates, and empty fields

The expected provider identifier is generally `Rndrng_NPI`; confirm the column exists before interpreting key checks. Exact duplicate rows and duplicate identifiers are separate checks.

In [ ]:
NPI_COL = "Rndrng_NPI"
TARGET = "Tot_Mdcr_Pymt_Amt"

print("Exact duplicate rows:", int(data.duplicated().sum()))
print("All-empty rows:", int(data.isna().all(axis=1).sum()))
print("All-empty columns:", int(data.isna().all(axis=0).sum()))

if NPI_COL in data.columns:
    print("Missing provider IDs:", int(data[NPI_COL].isna().sum()))
    print("Duplicate non-null provider IDs:", int(data.loc[data[NPI_COL].notna(), NPI_COL].duplicated().sum()))
else:
    print(f"WARNING: expected identifier {NPI_COL!r} is absent. Inspect the data dictionary and actual schema.")

if TARGET in data.columns:
    print("Missing target values:", int(data[TARGET].isna().sum()))
else:
    print(f"WARNING: expected target {TARGET!r} is absent. Check the file version and data dictionary.")

## 6. Missingness and potential suppression

Compute missing counts and percentages for all columns and visualize the fields with the highest missingness. Missing values can arise from suppression, non-applicability, or unavailable reporting; they are not necessarily data-entry errors. Consult the CMS dictionary before imputation. Do not fill missing or suppressed values with zero unless the source definition explicitly says that zero is the correct value.

In [ ]:
missingness = schema[["column", "missing_count", "missing_pct"]].sort_values("missing_pct", ascending=False)
display(missingness.head(30))
missingness.to_csv(OUTPUT_DIR / "missingness.csv", index=False)

top_missing = missingness.loc[missingness["missing_count"] > 0].head(25).sort_values("missing_pct")
if not top_missing.empty:
    plt.figure(figsize=(10, max(5, 0.28 * len(top_missing))))
    plt.barh(top_missing["column"], top_missing["missing_pct"])
    plt.xlabel("Missing values (%)")
    plt.title("Top 25 columns by missingness")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "missingness_top25.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("No missing values found.")

## 7. Numeric summaries and domain sanity checks

Review distribution summaries, negative values, zeros, and infinite values. A statistical outlier is only a flag for investigation, not proof of invalid data. Payment and utilization totals can be highly skewed and large values may be legitimate.

In [ ]:
numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
if numeric_cols:
    numeric_summary = data[numeric_cols].describe(percentiles=[.01, .05, .25, .50, .75, .90, .95, .99]).T
    numeric_summary["missing_count"] = data[numeric_cols].isna().sum()
    numeric_summary["zero_count"] = (data[numeric_cols] == 0).sum()
    numeric_summary["negative_count"] = (data[numeric_cols] < 0).sum()
    numeric_summary["infinite_count"] = np.isinf(data[numeric_cols]).sum()
    display(numeric_summary)
    numeric_summary.to_csv(OUTPUT_DIR / "numeric_summary.csv")
else:
    print("No numeric columns detected; inspect import dtypes and source formatting.")

## 8. Target distribution: total Medicare payment

This section uses the target only if it exists. Report median and upper percentiles alongside the mean because payment totals may be strongly right-skewed. Candidate high-cost thresholds are data-derived operational definitions for this project, not official CMS labels.

In [ ]:
if TARGET in data.columns:
    target = pd.to_numeric(data[TARGET], errors="coerce")
    target_valid = target.dropna()
    display(target_valid.describe(percentiles=[.50, .75, .90, .95, .99]).to_frame("total_medicare_payment"))

    print("Missing:", int(target.isna().sum()))
    print("Zeros:", int((target == 0).sum()))
    print("Negative values:", int((target < 0).sum()))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].hist(target_valid, bins=80)
    axes[0].set_title("Payment distribution (raw scale)")
    axes[0].set_xlabel("Total Medicare payment")
    axes[0].set_ylabel("Provider count")

    positive_target = target_valid[target_valid > 0]
    if not positive_target.empty:
        axes[1].hist(np.log1p(positive_target), bins=80)
        axes[1].set_title("Payment distribution (log1p scale; positive values)")
        axes[1].set_xlabel("log(1 + payment)")
        axes[1].set_ylabel("Provider count")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "target_payment_distributions.png", dpi=160, bbox_inches="tight")
    plt.show()

    thresholds = target_valid.quantile([.90, .95, .99]).rename("payment_threshold").to_frame()
    thresholds["providers_at_or_above"] = [int((target_valid >= t).sum()) for t in thresholds["payment_threshold"]]
    thresholds["observed_share_pct"] = thresholds["providers_at_or_above"] / len(target_valid) * 100
    display(thresholds)
    thresholds.to_csv(OUTPUT_DIR / "high_cost_threshold_candidates.csv")
else:
    print(f"Skipping target analysis because {TARGET!r} was not found.")

## 9. Categorical distributions: provider type and state

Use frequency tables and charts for the main categorical fields when present. These summaries help identify representation imbalances and possible stratification variables for RQ1. Categories with very small counts should be interpreted cautiously.

In [ ]:
for col in ["Rndrng_Prvdr_Type", "Rndrng_Prvdr_State_Abrvtn", "Rndrng_Prvdr_Mdcr_Prtcptg_Ind"]:
    if col in data.columns:
        counts = data[col].astype("string").fillna("<MISSING>").value_counts(dropna=False)
        display(counts.head(25).rename_axis(col).to_frame("provider_rows"))
        counts.head(20).sort_values().plot(kind="barh", figsize=(9, max(4, .25 * min(20, len(counts)))))
        plt.title(f"Top categories: {col}")
        plt.xlabel("Provider rows")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"category_{col}.png", dpi=160, bbox_inches="tight")
        plt.show()
    else:
        print(f"Column not found; skipped: {col}")

## 10. Bivariate analysis: payment by provider type

Compare provider-level payment distributions by provider type. Because payment totals are skewed and group sizes can differ substantially, show medians and quantiles rather than relying only on means. For readability, plot only the most common provider types. This is descriptive analysis; statistical tests and effect sizes should be selected after checking assumptions and defining the primary comparisons.

In [ ]:
if TARGET in data.columns and "Rndrng_Prvdr_Type" in data.columns:
    tmp = data[["Rndrng_Prvdr_Type", TARGET]].copy()
    tmp[TARGET] = pd.to_numeric(tmp[TARGET], errors="coerce")
    tmp = tmp.dropna(subset=[TARGET, "Rndrng_Prvdr_Type"])
    top_types = tmp["Rndrng_Prvdr_Type"].value_counts().head(12).index
    grouped = (tmp[tmp["Rndrng_Prvdr_Type"].isin(top_types)]
               .groupby("Rndrng_Prvdr_Type")[TARGET]
               .agg(count="count", mean="mean", median="median",
                    q75=lambda s: s.quantile(.75), q90=lambda s: s.quantile(.90))
               .sort_values("median", ascending=False))
    display(grouped)
    grouped.to_csv(OUTPUT_DIR / "payment_by_provider_type_summary.csv")

    plot_data = tmp[tmp["Rndrng_Prvdr_Type"].isin(top_types)].copy()
    order = plot_data.groupby("Rndrng_Prvdr_Type")[TARGET].median().sort_values(ascending=False).index
    # Boxplots use a log scale for visibility; zeros are retained and handled via symlog.
    fig, ax = plt.subplots(figsize=(12, 6))
    plot_data.boxplot(column=TARGET, by="Rndrng_Prvdr_Type", rot=70, ax=ax, grid=False)
    ax.set_yscale("symlog", linthresh=1)
    ax.set_title("Payment by common provider types (symlog scale)")
    ax.set_xlabel("Provider type")
    ax.set_ylabel("Total Medicare payment (symlog)")
    plt.suptitle("")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "payment_by_provider_type.png", dpi=160, bbox_inches="tight")
    plt.show()
else:
    print("Target or provider type column not found; skipped bivariate analysis.")

## 11. Correlation and target-leakage screening

Calculate Spearman correlations between the target and numeric columns. Correlation is not causal evidence. Very high correlations with payment-related measures can indicate that the candidate feature is mathematically or operationally close to the target. The primary prediction model should exclude direct/near-direct payment measures unless the prediction timing and business use case justify their availability.

In [ ]:
if TARGET in data.columns:
    corr_frame = data.select_dtypes(include=np.number).copy()
    if TARGET in corr_frame.columns:
        corr = corr_frame.corr(method="spearman", numeric_only=True)[TARGET].drop(TARGET, errors="ignore").dropna()
        corr = corr.reindex(corr.abs().sort_values(ascending=False).index)
        display(corr.head(30).to_frame("spearman_corr_with_target"))
        corr.to_frame("spearman_corr_with_target").to_csv(OUTPUT_DIR / "spearman_correlations_with_target.csv")

        plt.figure(figsize=(9, max(5, .25 * min(25, len(corr.head(25))))))
        corr.head(25).sort_values().plot(kind="barh")
        plt.xlabel("Spearman correlation with total Medicare payment")
        plt.title("Numeric features most associated with target (screening only)")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "target_spearman_correlations.png", dpi=160, bbox_inches="tight")
        plt.show()

    leakage_candidates = [
        "Tot_Mdcr_Alowd_Amt", "Tot_Mdcr_Stdzd_Amt",
        "Tot_Sbmtd_Chrg", "Tot_Srvcs"
    ]
    print("Review these potential leakage / timing-sensitive fields before modeling:")
    print([c for c in leakage_candidates if c in data.columns])
else:
    print("Target not found; skipped correlation analysis.")

## 12. Cross-field consistency checks

The checks below are conditional on the corresponding columns existing. CMS suppression and reporting rules can make component sums differ from totals. A discrepancy is a review flag, not automatic proof of an error. Always consult the official data dictionary before deciding how to handle it.

In [ ]:
def compare_sum_to_total(df, component_cols, total_col, label):
    available = [c for c in component_cols if c in df.columns]
    if total_col not in df.columns or len(available) < 2:
        print(f"Skipped {label}: required columns unavailable. Found components: {available}")
        return None
    comp = df[available].apply(pd.to_numeric, errors="coerce")
    total = pd.to_numeric(df[total_col], errors="coerce")
    complete = comp.notna().all(axis=1) & total.notna()
    diff = comp.sum(axis=1, min_count=len(available)) - total
    result = pd.DataFrame({
        "check": [label],
        "components_used": [", ".join(available)],
        "total_column": [total_col],
        "rows_with_complete_values": [int(complete.sum())],
        "rows_with_difference": [int((complete & diff.ne(0)).sum())],
        "rows_with_difference_pct_of_complete": [float((complete & diff.ne(0)).sum() / complete.sum() * 100) if complete.sum() else np.nan],
    })
    return result

checks = []
candidate_checks = [
    (["Bene_Age_0_64_Cnt", "Bene_Age_65_74_Cnt", "Bene_Age_75_84_Cnt", "Bene_Age_GT_84_Cnt"], "Tot_Benes", "Beneficiary age bands vs total beneficiaries"),
    (["Bene_Feml_Cnt", "Bene_Male_Cnt"], "Tot_Benes", "Sex counts vs total beneficiaries"),
    (["Bene_Dual_Cnt", "Bene_Ndual_Cnt"], "Tot_Benes", "Dual-status counts vs total beneficiaries"),
]
for comps, total_col, label in candidate_checks:
    result = compare_sum_to_total(data, comps, total_col, label)
    if result is not None:
        checks.append(result)

if checks:
    consistency = pd.concat(checks, ignore_index=True)
    display(consistency)
    consistency.to_csv(OUTPUT_DIR / "consistency_checks.csv", index=False)
else:
    print("No configured component-total checks could be run; verify actual column names against the CMS dictionary.")

## 13. Outlier review using IQR (flag only; do not automatically cap)

The supplied reference notebook uses the 1.5 × IQR rule and boxplots to flag potential outliers. For this Medicare project, use IQR only as a screening tool. Payment totals and utilization counts may have legitimate long right tails; automatic capping or deletion could remove the very high-cost providers the project is studying.

`IQR = Q3 - Q1`; lower flag = `Q1 - 1.5 × IQR`; upper flag = `Q3 + 1.5 × IQR`.

In [ ]:
outlier_rows = []
for col in numeric_cols:
    s = pd.to_numeric(data[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty or s.nunique() < 2:
        continue
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((s < low) | (s > high)).sum())
    outlier_rows.append({
        "column": col, "q1": q1, "q3": q3, "iqr": iqr,
        "lower_iqr_bound": low, "upper_iqr_bound": high,
        "flagged_count": n_out, "flagged_pct_of_nonmissing": n_out / len(s) * 100,
        "nonmissing_n": len(s)
    })
outlier_summary = pd.DataFrame(outlier_rows).sort_values("flagged_pct_of_nonmissing", ascending=False)
display(outlier_summary.head(30))
outlier_summary.to_csv(OUTPUT_DIR / "iqr_outlier_flags.csv", index=False)

# Boxplot of selected payment/utilization variables, if available.
plot_cols = [c for c in [TARGET, "Tot_Benes", "Tot_Srvcs", "Tot_HCPCS_Cds"] if c in data.columns]
if plot_cols:
    fig, axes = plt.subplots(len(plot_cols), 1, figsize=(10, 3 * len(plot_cols)))
    if len(plot_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, plot_cols):
        vals = pd.to_numeric(data[col], errors="coerce").dropna()
        ax.boxplot(vals, vert=False, showfliers=False)
        ax.set_title(f"{col} (boxplot hides fliers for readability; observations retained)")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "selected_numeric_boxplots.png", dpi=160, bbox_inches="tight")
    plt.show()

## 14. Build an EDA evidence summary

This summary records actual outputs from the loaded file. Use these outputs to write the project boundary and limitations; do not copy expected results from earlier runs if the file version differs.

In [ ]:
summary = {
    "input_file": str(DATA_PATH),
    "rows": int(data.shape[0]),
    "columns": int(data.shape[1]),
    "exact_duplicate_rows": int(data.duplicated().sum()),
    "target_column_present": TARGET in data.columns,
    "provider_id_column_present": NPI_COL in data.columns,
    "target_missing_count": int(data[TARGET].isna().sum()) if TARGET in data.columns else None,
    "target_zero_count": int((pd.to_numeric(data[TARGET], errors="coerce") == 0).sum()) if TARGET in data.columns else None,
    "top_10_missing_fields": missingness.head(10).to_dict(orient="records"),
}
if NPI_COL in data.columns:
    summary["provider_id_missing_count"] = int(data[NPI_COL].isna().sum())
    summary["duplicate_nonnull_provider_ids"] = int(data.loc[data[NPI_COL].notna(), NPI_COL].duplicated().sum())
if TARGET in data.columns:
    t = pd.to_numeric(data[TARGET], errors="coerce").dropna()
    summary["target_summary"] = {
        "mean": float(t.mean()), "median": float(t.median()),
        "p75": float(t.quantile(.75)), "p90": float(t.quantile(.90)),
        "p95": float(t.quantile(.95)), "p99": float(t.quantile(.99)),
        "max": float(t.max())
    }

with open(OUTPUT_DIR / "eda_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)
display(pd.json_normalize(summary, sep="_").T.rename(columns={0:"value"}))
print("Wrote:", OUTPUT_DIR / "eda_summary.json")

## 15. Interpretation template: boundaries and limitations

Complete this section after executing the notebook and reviewing the generated tables/plots.

### Data-supported boundaries
- **Unit of analysis:** provider/NPI aggregate record, not individual claim or patient.
- **Target:** `Tot_Mdcr_Pymt_Amt`, after confirming it exists and reviewing its CMS definition.
- **Population:** only the population represented by the specific CMS file/version.
- **High-cost definition:** choose a pre-specified percentile threshold (e.g., 90th percentile) and document that it is a research-defined threshold, not a CMS risk label.
- **Feature eligibility:** exclude direct payment components and any feature unavailable at the intended prediction time from the primary predictive model.

### Limitations to assess from actual outputs
- Missingness and privacy suppression may reduce usable sample size and distort subgroup comparisons.
- Top-coded chronic-condition percentages limit interpretation above the published cap.
- A single reporting year limits temporal generalization and cannot establish future-year predictive performance.
- Strong right skew and extreme values may influence squared-error metrics; report MAE and RMSE and consider a justified log-target sensitivity analysis.
- Provider-level aggregation prevents patient-level or claim-level conclusions.
- Association and predictive ranking do not establish causality, fraud, waste, abuse, or provider quality.

### Data quality decisions
Document each decision in a table with: issue, evidence/output file, decision, justification, and impact on RQ1–RQ4. Do not drop or winsorize records solely because the IQR rule flags them. Preserve a raw copy and create separate modeling data after decisions are documented.

## 16. Export checklist

The notebook exports tables and figures under `reports/eda/`. Confirm the outputs exist after execution, inspect them before reporting findings, and record the source file/version and run date.

Recommended repository files:
- `notebooks/01_data_quality_eda.ipynb`
- `data/README.md` with official CMS links and download instructions
- `reports/eda/` containing summary tables and charts (do not publish raw provider data unless permitted)
- `requirements.txt`

### Sources
- Centers for Medicare & Medicaid Services. Medicare Physician & Other Practitioners — by Provider, data landing page and data dictionary (links above).
- The supplied `Capstone_ExploratoryDataAnalysis.ipynb` informed the notebook organization: context, data description, sanity checks, missingness, summary statistics, distributions, outlier screening, and interpretation. The original Melbourne housing dataset and its domain assumptions were not carried over.